# 1.Human In The Loop

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command, interrupt


# 声明状态
class State(TypedDict):
    username: str


# 声明节点
def node_a(state: State) -> dict:
    username = interrupt("请输入你的名字")
    return {
        "username": username,
    }


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("node_a", node_a)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

# 使用中断必须设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)


In [ ]:
# 恢复执行
msg = res['__interrupt__'][0].value
username = input(msg)

resume_res = graph.invoke(Command(resume=username), config=config)
print(resume_res)